# 10 – Explorer

Interactive view of the datasets built by notebooks `01` and `02`. Run all cells, then use the widgets.

* **Company** – trends of selected metrics for one company
* **Compare** – one metric across several companies
* **Text** – 10-K text signals + newly added sentences for a selected year (needs notebook 02)
* **Snapshot** – latest fiscal year overview

In [ ]:
import html
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from stockpred.config import load_settings
from stockpred.plots import LABELS, SECTION_LABELS, trend_figure, compare_figure, snapshot_table, text_trend_figure
from stockpred.text import load_section, new_sentences

settings = load_settings()
P = settings.processed_dir
panel = pd.read_parquet(P / "fundamentals_annual.parquet")
universe = pd.read_csv(P / "universe.csv")
has_text = (P / "text_features.parquet").exists()
if has_text:
    text = pd.read_parquet(P / "text_features.parquet")
    filings = pd.read_parquet(P / "text_filings.parquet")
SECTIONS_DIR = settings.data_dir / "interim" / "sections"
print(f"{panel['ticker'].nunique()} companies, FY{panel['fiscal_year'].min()}–{panel['fiscal_year'].max()}, "
      f"text data: {'yes' if has_text else 'no – run 02_text_analysis.ipynb'}")

In [ ]:
# --- Company & Compare tabs ------------------------------------------------------
ticker_options = [(f"{r.ticker} – {r.name}", r.ticker) for r in universe.itertuples()]
metric_options = [(label, key) for key, label in LABELS.items() if key in panel.columns]
default_metrics = ["revenue", "net_income", "free_cash_flow", "eps_diluted",
                   "gross_margin", "operating_margin", "roe", "revenue_growth"]

ticker_dd = widgets.Dropdown(options=ticker_options, description="Company", layout={"width": "420px"})
metrics_sel = widgets.SelectMultiple(options=metric_options, value=default_metrics,
                                     description="Metrics", rows=10, layout={"width": "420px"})
company_out = widgets.Output()

compare_sel = widgets.SelectMultiple(options=ticker_options, value=[t for _, t in ticker_options[:5]],
                                     description="Companies", rows=10, layout={"width": "420px"})
compare_metric = widgets.Dropdown(options=metric_options, value="operating_margin", description="Metric")
compare_out = widgets.Output()


def draw_company(*_):
    with company_out:
        company_out.clear_output(wait=True)
        trend_figure(panel, ticker_dd.value, list(metrics_sel.value)).show()


def draw_compare(*_):
    with compare_out:
        compare_out.clear_output(wait=True)
        compare_figure(panel, list(compare_sel.value), compare_metric.value).show()


for w in (ticker_dd, metrics_sel):
    w.observe(draw_company, names="value")
for w in (compare_sel, compare_metric):
    w.observe(draw_compare, names="value")

In [ ]:
# --- Text tab ------------------------------------------------------------------
text_ticker = widgets.Dropdown(options=ticker_options, description="Company", layout={"width": "420px"})
text_section = widgets.ToggleButtons(options=[(v, k) for k, v in SECTION_LABELS.items()], description="Section")
text_year = widgets.Dropdown(description="Fiscal year")
text_chart_out, sentences_out = widgets.Output(), widgets.Output()


def update_years(*_):
    if not has_text:
        return
    years = sorted(filings.loc[filings["ticker"] == text_ticker.value, "fiscal_year"].unique())
    text_year.options = years[1:]                      # first year has nothing to compare with
    text_year.value = years[-1] if len(years) > 1 else None


def draw_text_chart(*_):
    with text_chart_out:
        text_chart_out.clear_output(wait=True)
        if has_text:
            text_trend_figure(text, text_ticker.value).show()
        else:
            print("Run notebooks/02_text_analysis.ipynb first.")


def draw_sentences(*_):
    with sentences_out:
        sentences_out.clear_output(wait=True)
        if not has_text or text_year.value is None:
            return
        f = filings[filings["ticker"] == text_ticker.value].set_index("fiscal_year").sort_index()
        years = list(f.index)
        cur = f.loc[text_year.value]
        prev = f.loc[years[years.index(text_year.value) - 1]]
        cur_txt = load_section(SECTIONS_DIR, cur.cik, cur.accession, text_section.value)
        prev_txt = load_section(SECTIONS_DIR, prev.cik, prev.accession, text_section.value)
        if not cur_txt or not prev_txt:
            print("Section not extracted for one of the two years.")
            return
        added = new_sentences(cur_txt, prev_txt)
        items = "".join(f"<li>{html.escape(s)}</li>" for s in added[:40])
        display(widgets.HTML(f"<b>{len(added)} new sentences</b> in FY{text_year.value} "
                             f"(showing up to 40)<ul style='line-height:1.4'>{items}</ul>"))


text_ticker.observe(lambda *_: (update_years(), draw_text_chart()), names="value")
for w in (text_year, text_section):
    w.observe(draw_sentences, names="value")

In [ ]:
# --- assemble tabs ---------------------------------------------------------------
snapshot_out = widgets.Output()
with snapshot_out:
    display(snapshot_table(panel, universe).style.format({
        "revenue_usd_bn": "{:,.1f}", "market_cap_usd_bn": "{:,.0f}",
        "revenue_cagr_5y": "{:.1%}", "gross_margin": "{:.1%}", "operating_margin": "{:.1%}",
        "net_margin": "{:.1%}", "fcf_margin": "{:.1%}", "roe": "{:.1%}",
    }, na_rep="–"))

tabs = widgets.Tab(children=[
    widgets.VBox([widgets.HBox([ticker_dd, metrics_sel]), company_out]),
    widgets.VBox([widgets.HBox([compare_sel, compare_metric]), compare_out]),
    widgets.VBox([widgets.HBox([text_ticker, text_year]), text_chart_out, text_section, sentences_out]),
    snapshot_out,
])
for i, title in enumerate(["Company", "Compare", "Text", "Snapshot"]):
    tabs.set_title(i, title)

draw_company(); draw_compare(); update_years(); draw_text_chart(); draw_sentences()
display(tabs)

### How to read the charts
* Fundamentals are **as first reported** in each 10-K (later restatements ignored) – what investors knew at the time.
* EPS is split-adjusted to today's share basis. ROE > 100 % happens with tiny equity after buybacks (e.g. Apple).
* **Text:** tone of Risk Factors is always strongly negative (by design) – watch the *change*, not the level.
  *New sentences* = sentences without a ≥ 80 % word match in any sentence of the previous year.